# 0.  Introduction to PlatoSim

Welcome to PlatoSim! In this introductory Jupyter notebook tutorial we show how you can get started generating simulations. Furthermore we give an overview of all the utilies and help functions that are build into PlatoSim to help you getting started - enjoy!

### Setup notebook

In [ ]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# Configure figure in notebook
%matplotlib notebook

### Imports

In [ ]:
import os
import h5py
import yaml
import pyaml
import numpy as np
import matplotlib.pyplot as plt
from yaml.loader import SafeLoader
from prettytable import PrettyTable

# PlatoSim
from platosim.simulation import Simulation
from platosim.simfile    import SimFile

from platosim.matplotlibrc import setup
setup()

---
## 0.1 - Setup and run a simulation
---

As part of the installation of PlatoSim the project directory and working directory should already have been set. We don't need these paths for this series of tutorials, however, they are very handy when starting create your own scripts. Thus check that these are globally defined:

In [ ]:
projectDir = os.environ['PLATO_PROJECT_HOME']
workDir    = os.environ['PLATO_WORKDIR']
print(projectDir)
print(workDir)

First we define the I/O paths and files needed for PlatoSim.

In [ ]:
# We use the default YAML input file and configure is later
inputDir  = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles"
inputFile = inputDir + "/inputfile.yaml"

# Save all output to current working directory
outputDir = os.getcwd()

Now a simulation object can be created. Note that by default PlatoSim used the YAML input file located at `PLATO_PROJECT_HOME`, however, we here show how to include it directly for the construction of the simulation object: 

In [ ]:
# Set up a Simulation object
outputFileName = "output_example1"
sim = Simulation(outputFileName, inputFile, outputDir=outputDir)

Note that you can set the output directory afterwards as well using 

```
sim.outputDir = outputDir
```

Also if not specified, the `inputfile.yaml` configuration file from the `/inputfiles` directory will be used (provided you have exported the `PLATO_PROJECT_HOME` environment variable). It is possible to also set the path to the reference input file as follows: 

```
sim.readConfigurationFile(<full path to the configuration file>)
```

However, for both parameters, choose only one method since setting both will crash PlaotSim upon execution.

For now we use the default YAML settings to run our simulation. Note that by default PlatoSim do not overwrite the output from a previous simulation defined by the same name. To avoid that PlatoSim ends in errors, we here activate the `removeOutputFile` option when launching PlatoSim: 

In [ ]:
# Run PlatoSim
simFile = sim.run(removeOutputFile=True)

**Congratulations, you have a now generated your first PlatoSim simulation with Python!**

In this directory, the following information will be stored when running the simulation:
- `runName.yaml`: copy of the input file with the chosen configuration parameters (copied from the reference configuration file and possibly modified, as shown below);
- `runName.hdf5`: resulting exposures (images, PSF, etc.);
- `runName.log`: log file (to report any problems).


---
## 0.2 - Configuring the YAML input file
---

We note that all the configuration parameters in the YAML input file is the topic of the following tutorial: **01_ConfigurationParametersYAML**. Now say that we want to configure the input parameters but we don't want to open up the YAML input file to do so (e.g. because we want to make an automated script for our simulations). Again first we create an simulation object:

In [ ]:
# Initialise PlatoSim
outputFileName = "output_example2"
sim = Simulation(outputFileName, outputDir=outputDir)

Since we are persistent and really don't want to open the YAML file manually, let's have a look at it's content from within Python. Here we use the `SafeLoader` option of the `yaml` package, while we print (or dump) the data more nicely with the `pyaml` package: 

In [ ]:
# Open and print content
with open(inputFile) as f:
    data = yaml.load(f, Loader=SafeLoader)
    print(pyaml.dump(data))

Note that `pyaml` outputs everything alphabetically, however, the real YAML input file has a different structure (so don't panic). After an overview of the input parameters let's assume we want to change the following parameters:

In [ ]:
# Observation
sim["ObservingParameters/NumExposures"] = 10

# Sky
sim["Sky/SkyBackground"]         = -1
sim["Sky/Cosmics/CosmicHitRate"] = 10

# Subfield
sim["SubField/NumColumns"]      = 300
sim["SubField/NumRows"]         = 300
sim["SubField/ZeroPointColumn"] = 0
sim["SubField/ZeroPointRow"]    = 0

# Control HDF5
sim["ControlHDF5Content/WriteStarPositions"] = True

We have chosen to simulate only 1 image, set the sky background to be estimated from an automatic tabulation according to the sky location (done with the `-1`), and we reduced the cosmic ray hit rate to "quiet" space weather conditions for PLATO (according to the results of the PLATO working group). The subfield is enlarged but we keep the simulation in the origo-corner of the CCD. We have likewise secured a identical CCD gain for the E and F side of each of the 4 CCDs in the camera focal plane. Let's run the simulation:

In [ ]:
# Run PlatoSim
simfile = sim.run(removeOutputFile=True)

---
## 0.3 - Fetching the HDF5 output file
---

To access the HDF5 output file that contains your simulated data, we can simply use the `SimFile` class:

In [ ]:
# Fetch HDF5 output
f = SimFile(outputFileName + ".hdf5")

---
## 0.4 - Plot the simulated subfield
---

Plotting your simulated pixel images can be done out-of-box using the `showImage` utility of the `SimFile` class. This function takes can plot either a specific frame using `imageNr` or the entire image cube with an interactive slicer below the plot as follows:

In [ ]:
fig, ax = f.showImage(imageNr=False, clipPercentile=1, imgScale="clip",  
                      figsize=(7,7), fontSize=15, useTitle=False,
                      showStarPositions=False, showStarIDs=False,
                      colorMap="cubehelix", colorBar=True, showGrid=True) 

---
## 0.5 - Overview of PlatoSim's class functions
---

We have now covered the basics of using PlatoSim within a Python interface. The following is an overview of all the help functions and utilities that are directly build into the PlatoSim software package to ease your life of running, analyzing, and exploring your simulations. Below each Python class is a file located in the `python/platosim/` directory.

The utilities of the `Simulation` class can be used to **prepare and setup** your simulation:

In [ ]:
t = PrettyTable()
t.add_column("Simulation functions", dir(Simulation)[29:])
t

The utilities of the `SimFile` class can be used to **retrieve and save information** about your simulation:

In [ ]:
t = PrettyTable()
t.add_column("SimFile functions", dir(SimFile)[27:])
t

The utilities of the `Photometry` class can be used to **retrieve and analyze the photometry** about your simulation:

In [ ]:
from platosim.photometry import Photometry
t = PrettyTable()
t.add_column("Photometry functions", dir(Photometry)[27:])
t

The utilities of the `referenceFrames` script are **coordinate transformations** used by PlatoSim:

In [ ]:
import platosim.referenceFrames as rf 
t = PrettyTable()
t.add_column("referenceFrames functions", dir(rf))
t

---
## 0.6 - Overview of PlatoSim's  utility functions
---

The functions of the `utilities` script are **help functions** used by PlatoSim:

In [ ]:
import platosim.utilities as ut 
t = PrettyTable()
t.add_column("referenceFrames functions", dir(ut))
t

The functions of the `plot` script are **graphical functions** used by PlatoSim:

In [ ]:
import platosim.plot as pt 
t = PrettyTable()
t.add_column("referenceFrames functions", dir(pt))
t